In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [9]:
from embedder import Embedder

In [10]:
embedder_ = Embedder()

Q1

In [15]:
v1 = embedder_.encode('How does approximate nearest neighbor search work?')

In [16]:
v1[0]

np.float64(-0.02058203437252893)

Q2

In [18]:
doc2 = [doc['content'] for doc in documents if doc['filename'] == '02-vector-search/lessons/07-sqlitesearch-vector.md'][0]

In [20]:
v2 = embedder_.encode(doc2)

In [21]:
v1.dot(v2)

np.float64(0.36107027225589694)

Q3

In [24]:
import numpy as np

In [22]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [31]:
chunks_content = [chunk['content'] for chunk in chunks]

In [35]:
X =  embedder_.encode_batch(chunks_content)

In [36]:
scores = X.dot(v1)

In [37]:
most_relevant_id = np.argmax(scores)

In [39]:
chunks[most_relevant_id]['filename']

'02-vector-search/lessons/07-sqlitesearch-vector.md'

Q4

In [40]:
from minsearch import VectorSearch

In [42]:
vindex = VectorSearch(keyword_fields=['content'])
vindex.fit(X, chunks)

In [46]:
vindex.search(embedder_.encode('What metric do we use to evaluate a search engine?'), num_results=1)[0]['filename']

'04-evaluation/lessons/05-search-metrics.md'

Q5

In [48]:
from minsearch import Index

In [53]:
index = Index(text_fields=['content'])

In [54]:
index.fit(chunks)

In [ ]:
vector_results = vindex.search(embedder_.encode('How do I store vectors in PostgreSQL?'), num_results=5)

In [56]:
index_results = index.search('How do I store vectors in PostgreSQL?', num_results=5)

In [60]:
set(vector_result['filename'] for vector_result in vector_results) - set(index_result['filename'] for index_result in index_results)

{'02-vector-search/lessons/08-pgvector.md'}

Q6

In [61]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [62]:
vector_results_6 = vindex.search(embedder_.encode('How do I give the model access to tools?'), num_results=5)
index_results_6 = index.search('How do I give the model access to tools?', num_results=5)

In [63]:
results = rrf([vector_results_6, index_results_6])

In [65]:
results[0]['filename']

'01-agentic-rag/lessons/13-function-calling.md'